# 02. LangChain 제품 계층: Framework vs Runtime vs Harness

> LangChain 생태계는 **Framework(프로토타입) → Runtime(제어) → Harness(완성된 에이전트)** 의 3계층으로 나뉘어요. 각 계층이 푸는 문제와 선택 기준을 비교해요.

## 학습 목표

이 노트북을 마치면 다음을 할 수 있어요:

1. LangChain(Framework), LangGraph(Runtime), Deep Agents(Harness) 세 계층의 역할을 구분할 수 있어요
2. 각 계층이 어떤 문제를 해결하기 위해 만들어졌는지 설명할 수 있어요
3. 동일한 목표(날씨 에이전트)를 세 가지 방식으로 구현하고 차이점을 비교할 수 있어요
4. 실제 프로젝트에서 어느 계층을 선택해야 할지 판단 기준을 갖출 수 있어요

## 사전 지식

- 이전 노트북 `01-LangChain-V1-Overview.ipynb`: V1 철학 변화, `init_chat_model`, `create_agent`
- Python 함수 데코레이터 기본 이해
- LLM 도구 호출(Tool Calling) 개념


## LangChain 제품 계층이란?

LangChain 생태계는 **세 가지 독립적이면서도 상호 연결된 계층**으로 이루어져 있어요. 각 계층은 서로 다른 수준의 추상화를 제공하고, 서로 다른 문제를 해결해요.

### 왜 세 가지 계층이 필요할까요?

에이전트를 개발할 때 겪는 고민을 생각해봐요:

- **"빠르게 프로토타입을 만들고 싶다"** → 높은 추상화가 필요해요
- **"복잡한 상태를 관리하고, 오류 복구와 Human-in-the-Loop를 구현해야 한다"** → 저수준 제어가 필요해요  
- **"코드 에디터처럼 파일시스템을 다루고, 서브에이전트를 띄우는 실제 에이전트가 필요하다"** → 완전한 하네스가 필요해요

LangChain은 이 세 가지 요구사항 각각을 위한 계층을 제공해요.

> 🔑 **핵심 개념**: 세 계층은 **선택지**예요, 의무적인 단계가 아니에요. 프로젝트 복잡도에 따라 적절한 계층을 선택하거나, 조합해서 사용할 수 있어요.

### 비유로 이해하기: 요리와 비교해볼까요?

세 계층을 요리에 비유하면 이해가 쉬워요:

| 계층 | 요리 비유 | 설명 |
|------|-----------|------|
| **LangChain Framework** | 밀키트(Meal Kit) | 재료와 레시피가 준비되어 있어요. 순서대로 따라하면 요리가 완성돼요. |
| **LangGraph Runtime** | 주방 도구 세트 | 칼, 도마, 프라이팬 등 개별 도구를 직접 선택해서 나만의 레시피를 만들어요. |
| **Deep Agents Harness** | 완전 자동화 주방 | 냉장고 관리, 레시피 검색, 조리까지 주방 전체가 시스템으로 돌아가요. |

밀키트로 시작해서, 더 세밀한 조리가 필요하면 직접 도구를 쓰고, 대규모 자동화가 필요하면 자동화 주방을 도입하는 거예요.


## 전체 아키텍처

```mermaid
flowchart TB
    subgraph HA["Deep Agents (Harness) — 배터리 포함, 완전한 에이전트 환경"]
        direction TB
        H1["write_todos<br>할 일 관리"]
        H2["filesystem<br>파일 읽기/쓰기"]
        H3["subagents<br>서브에이전트 위임"]
        H4["code_execution<br>코드 실행"]
        H5["HITL<br>사람 승인"]
        H6["context<br>컨텍스트 압축"]
    end

    subgraph RT["LangGraph (Runtime) — 저수준 오케스트레이션 엔진"]
        direction TB
        R1["StateGraph<br>상태 그래프"]
        R2["Checkpointer<br>체크포인트"]
        R3["Streaming<br>스트리밍"]
        R4["interrupt<br>Human-in-the-Loop"]
    end

    subgraph FW["LangChain (Framework) — 통합 + 추상화 레이어"]
        direction TB
        F1["init_chat_model<br>통합 모델 초기화"]
        F2["create_agent<br>에이전트 생성"]
        F3["@tool<br>도구 정의"]
        F4["100+ 통합<br>(OpenAI, Anthropic...)"]
    end

    HA --> RT
    RT --> FW
    FW --> LLM["LLM 공급자<br>(OpenAI, Anthropic, Google...)"]

    classDef harness fill:#f8d7da,stroke:#dc3545,color:#721c24
    classDef runtime fill:#cce5ff,stroke:#007bff,color:#004085
    classDef framework fill:#d4edda,stroke:#28a745,color:#155724
    classDef llm fill:#fff3cd,stroke:#ffc107,color:#856404

    class H1,H2,H3,H4,H5,H6 harness
    class R1,R2,R3,R4 runtime
    class F1,F2,F3,F4 framework
    class LLM llm
```

### 세 계층 한눈에 비교

| 계층 | 제품 | 추상화 수준 | 주요 특징 | 선택 시점 |
|------|------|------------|----------|----------|
| **Framework** | LangChain | 높음 | 통합, 도구, `create_agent` | 빠른 프로토타입, 표준 에이전트 |
| **Runtime** | LangGraph | 낮음 | `StateGraph`, 체크포인트, HITL | 복잡한 워크플로우, 세밀한 제어 |
| **Harness** | Deep Agents | 최고 | 파일시스템, 서브에이전트, 6가지 기능 | 실제 코딩 에이전트, 장기 실행 태스크 |


## 1. 환경 설정


In [ ]:
# ---------------------------------------------------
# 환경 변수 로드
# ---------------------------------------------------
# .env 파일에서 OPENAI_API_KEY 등을 읽어와요
from dotenv import load_dotenv

load_dotenv()


In [ ]:
from langchain.tools import tool

# TODO: 이 셀의 핵심 구현 코드를 수업 시간에 직접 작성하세요.
# 목표: 공용 도구 정의: 세 계층 비교에서 동일하게 사용해요
# 위 마크다운 설명과 힌트를 참고해 필요한 타입, 함수, 그래프 구성, 실행 코드를 완성하세요.


## 2. 계층 1 — LangChain Framework

LangChain Framework는 **통합(Integration) + 추상화(Abstraction)** 레이어예요.

### LangChain이 해결하는 문제

- OpenAI, Anthropic, Google 등 수십 개의 모델 공급자가 각각 다른 API를 가지고 있어요
- 벡터 스토어, 리트리버, 문서 로더도 모두 다른 인터페이스를 가지고 있어요
- LangChain은 이것들을 **하나의 통일된 인터페이스**로 추상화해요

### LangChain의 역할

| 기능 | 예시 |
|------|----- |
| 모델 통합 | `init_chat_model("openai:gpt-4o-mini")` |
| 도구 정의 | `@tool` 데코레이터 |
| 에이전트 생성 | `create_agent(model, tools, system_prompt)` |
| 100+ 통합 | OpenAI, Anthropic, FAISS, Pinecone... |


In [ ]:
from langchain.agents import create_agent
from langchain.chat_models import init_chat_model
from langchain.messages import HumanMessage

model = init_chat_model("openai:gpt-4o-mini")

# TODO: 이 셀의 핵심 구현 코드를 수업 시간에 직접 작성하세요.
# 목표: 계층 1: LangChain Framework로 날씨 에이전트 구현
# 위 마크다운 설명과 힌트를 참고해 필요한 타입, 함수, 그래프 구성, 실행 코드를 완성하세요.


In [ ]:
from IPython.display import Image, display

# TODO: 이 셀의 핵심 구현 코드를 수업 시간에 직접 작성하세요.
# 목표: 그래프 흐름: START → agent → tools → agent → ... → END
# 위 마크다운 설명과 힌트를 참고해 필요한 타입, 함수, 그래프 구성, 실행 코드를 완성하세요.


### LangChain Framework 요약

위 코드를 보면, 단 몇 줄로 도구를 사용하는 에이전트를 만들었어요. 이것이 LangChain Framework의 강점이에요:
- 복잡한 설정 없이 빠르게 시작
- 모든 공급자에 통일된 인터페이스

하지만 **한계**도 있어요: 에이전트가 실패했을 때 어떻게 복구할지, 중간 상태를 저장하려면 어떻게 해야 하는지, 사람이 개입하려면 어떻게 해야 하는지... 이런 제어가 필요하면 다음 레이어인 LangGraph가 필요해요.


## 3. 계층 2 — LangGraph Runtime

LangGraph는 **저수준 오케스트레이션 엔진**이에요. LangChain Framework가 내부적으로 사용하는 바로 그 엔진이에요.

### LangGraph가 해결하는 문제

- **상태 지속성**: 긴 대화나 복잡한 작업에서 중간 상태를 저장하고 복원해야 해요
- **오류 복구**: 특정 단계가 실패했을 때 그 지점부터 재시작해야 해요
- **Human-in-the-Loop**: 특정 단계에서 사람의 승인을 기다려야 해요
- **복잡한 조건 분기**: 단순한 선형 흐름이 아닌 복잡한 워크플로우가 필요해요

### LangGraph의 핵심 개념

```mermaid
flowchart LR
    S(["START"]) --> N1["노드 1<br>(Node)"]  
    N1 -->|"엣지 (Edge)"| N2["노드 2"]
    N2 -->|"조건부 엣지<br>(Conditional Edge)"| N3["노드 3"]
    N2 -->|"조건부 엣지"| N4["노드 4"]
    N3 --> E(["END"])
    N4 --> E

    subgraph ST["State (상태)"]
        direction TB
        ST1["messages: list"]
        ST2["other_fields: any"]
    end

    N1 -.->|"상태 읽기/쓰기"| ST
    N2 -.->|"상태 읽기/쓰기"| ST

    classDef node fill:#cce5ff,stroke:#007bff,color:#004085
    classDef state fill:#e2d5f1,stroke:#6f42c1,color:#3d1f6e
    classDef terminal fill:#d4edda,stroke:#28a745,color:#155724

    class N1,N2,N3,N4 node
    class ST,ST1,ST2 state
    class S,E terminal
```

| 개념 | 설명 |
|------|------|
| **StateGraph(상태 그래프)** | 노드와 엣지로 이루어진 실행 그래프예요 |
| **State(상태)** | 그래프 전체에서 공유하는 데이터 컨테이너예요 |
| **Node(노드)** | 상태를 읽고 수정하는 함수예요 |
| **Edge(엣지)** | 노드 간 연결이에요 (조건부도 가능) |
| **Checkpointer(체크포인터)** | 각 실행 단계를 저장해서 복구와 Time Travel을 가능하게 해요 |


In [ ]:
from typing import Annotated
from typing_extensions import TypedDict
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages
from langgraph.prebuilt import ToolNode, tools_condition

model_with_tools = init_chat_model("openai:gpt-4o-mini").bind_tools(tools)

# TODO: 이 셀의 핵심 구현 코드를 수업 시간에 직접 작성하세요.
# 목표: 계층 2: LangGraph Runtime으로 날씨 에이전트 구현
# 위 마크다운 설명과 힌트를 참고해 필요한 타입, 함수, 그래프 구성, 실행 코드를 완성하세요.


In [ ]:
from langchain.messages import HumanMessage
from IPython.display import Image, display

# TODO: 이 셀의 핵심 구현 코드를 수업 시간에 직접 작성하세요.
# 목표: 그래프 흐름: START → call_model → tools → call_model → ... → END
# 위 마크다운 설명과 힌트를 참고해 필요한 타입, 함수, 그래프 구성, 실행 코드를 완성하세요.


In [ ]:
# TODO: 이 셀의 핵심 구현 코드를 수업 시간에 직접 작성하세요.
# 목표: LangGraph 에이전트 실행 - 스트리밍 방식
# 위 마크다운 설명과 힌트를 참고해 필요한 타입, 함수, 그래프 구성, 실행 코드를 완성하세요.


### LangGraph Runtime 요약

LangChain Framework 버전보다 코드가 길어졌지만, **무슨 일이 일어나는지 완전히 제어**할 수 있어요:

- 어떤 노드가 실행되는지 명시적으로 확인할 수 있어요
- 중간에 체크포인터를 추가해서 상태를 저장할 수 있어요
- `interrupt`를 사용해서 특정 단계에서 사람의 승인을 받을 수 있어요

이 교재에서 Part 2~4에 걸쳐 LangGraph를 깊이 배울 거예요. 지금은 "더 세밀한 제어를 위한 엔진"이라는 개념만 이해해도 충분해요.

그럼 LangGraph보다 더 높은 수준의 추상화가 필요한 경우는 어떤 경우일까요? 파일을 읽고 쓰고, 서브에이전트를 띄우고, 코드를 실행하는 **실제 시스템과 상호작용하는 에이전트**가 필요할 때예요. 이것이 바로 다음에 살펴볼 Deep Agents의 영역이에요.


## 4. 계층 3 — Deep Agents Harness

Deep Agents는 **배터리 포함(batteries-included)** 에이전트 하네스예요.

### Deep Agents가 해결하는 문제

Claude Code, Devin, GitHub Copilot Workspace 같은 코딩 에이전트를 상상해봐요. 이런 에이전트는:

- 파일을 읽고 수정해야 해요
- 할 일 목록을 관리해야 해요
- 복잡한 태스크를 서브에이전트에 위임해야 해요
- 코드를 직접 실행해야 해요
- 장기간 실행 중에 사람의 승인을 받아야 해요
- 긴 컨텍스트를 압축해서 관리해야 해요

이것들을 LangGraph로 직접 구현하는 것은 복잡해요. Deep Agents는 이런 **공통 패턴을 미리 구현**해서 제공해요.

### Deep Agents의 6가지 Harness 기능

| 기능 | 설명 | 예시 |
|------|------|------|
| **Planning** | 할 일 목록 자동 생성 및 관리 | `write_todos(tasks)` |
| **Filesystem** | 파일 읽기/쓰기/수정 | `read_file(path)`, `write_file(path, content)` |
| **Subagents** | 전문화된 서브에이전트에 위임 | `create_subagent(instructions)` |
| **Context** | 장기 대화 컨텍스트 압축 | 자동 요약 및 압축 |
| **Code Execution** | 코드 직접 실행 | Python, Bash 실행 환경 |
| **HITL** | 중요한 작업 전 사람 승인 요청 | `request_approval(action)` |


### Deep Agents 코드 패턴 (참고용 pseudocode)

Deep Agents의 실제 사용 패턴을 간단히 살펴봐요. 실제 사용은 Part 10에서 자세히 다뤄요.

```python
# 실제 Deep Agents 코드 패턴 (Part 10에서 배워요)
from langchain.agents import create_deep_agent

# 하네스 기능을 갖춘 에이전트 생성
deep_agent = create_deep_agent(
    model=init_chat_model("openai:gpt-4o-mini"),
    harness=[
        "planning",       # 할 일 목록 자동 관리
        "filesystem",     # 파일 읽기/쓰기
        "subagents",      # 서브에이전트 위임
        "code_execution", # 코드 실행
    ],
    system_prompt="당신은 날씨 데이터를 분석하고 보고서를 작성하는 에이전트예요.",
)

# 복잡한 태스크 실행
result = deep_agent.run(
    "서울, 부산, 제주의 날씨를 조사하고"
    " weather_report.md 파일에 보고서를 작성해줘"
)
# Deep Agents는 자동으로:
# 1. 할 일 목록 생성 (도시별 날씨 조회, 보고서 작성)
# 2. 각 도시 날씨 도구 호출
# 3. 수집된 데이터를 파일에 작성
# 4. 완료 보고
```


In [ ]:
import json
from datetime import datetime

# TODO: 이 셀의 핵심 구현 코드를 수업 시간에 직접 작성하세요.
# 목표: Deep Agents 핵심 아이디어: Planning + Filesystem 시뮬레이션
# 위 마크다운 설명과 힌트를 참고해 필요한 타입, 함수, 그래프 구성, 실행 코드를 완성하세요.


## 5. 세 계층 비교: 언제 무엇을 선택할까요?

### Feature Comparison

| 기능 | LangChain Framework | LangGraph Runtime | Deep Agents Harness |
|------|:-------------------:|:-----------------:|:-------------------:|
| 빠른 시작 | ✅ 매우 쉬움 | 보통 | 보통 |
| 상태 지속성 | 기본 지원 | ✅ 완전 제어 | ✅ 자동 관리 |
| 스트리밍 | ✅ 지원 | ✅ 완전 제어 | ✅ 자동 |
| Human-in-the-Loop | 제한적 | ✅ `interrupt` | ✅ 내장 |
| 파일시스템 접근 | 도구로 직접 구현 | 도구로 직접 구현 | ✅ 내장 |
| 서브에이전트 | 제한적 | 수동 구현 | ✅ 내장 |
| 세밀한 제어 | 제한적 | ✅ 완전 제어 | 추상화됨 |
| 학습 곡선 | 낮음 | 보통 | 보통 |

### 선택 가이드

```mermaid
flowchart TD
    Q1{"파일 시스템, 서브에이전트,
    코드 실행이 필요한가요?"}
    Q2{"복잡한 상태 관리, HITL,
    오류 복구가 필요한가요?"}
    A1["Deep Agents Harness<br>배터리 포함 에이전트"]
    A2["LangGraph Runtime<br>저수준 오케스트레이션"]
    A3["LangChain Framework<br>빠른 시작"]

    Q1 -->|Yes| A1
    Q1 -->|No| Q2
    Q2 -->|Yes| A2
    Q2 -->|No| A3

    classDef harness fill:#f8d7da,stroke:#dc3545,color:#721c24
    classDef runtime fill:#cce5ff,stroke:#007bff,color:#004085
    classDef framework fill:#d4edda,stroke:#28a745,color:#155724
    classDef question fill:#fff3cd,stroke:#ffc107,color:#856404

    class A1 harness
    class A2 runtime
    class A3 framework
    class Q1,Q2 question
```


In [ ]:
from langchain.messages import HumanMessage

# TODO: 이 셀의 핵심 구현 코드를 수업 시간에 직접 작성하세요.
# 목표: 세 계층 비교: 동일한 질문에 대한 응답 비교
# 위 마크다운 설명과 힌트를 참고해 필요한 타입, 함수, 그래프 구성, 실행 코드를 완성하세요.


## 6. 경쟁 프레임워크와의 비교

LangChain 생태계 외에도 에이전트 개발을 위한 다양한 프레임워크가 있어요. 각각의 포지셔닝을 이해하면 기술 선택에 도움이 돼요.

| 프레임워크 | 주요 특징 | LangChain 생태계 대응 |
|------------|----------|----------------------|
| **Vercel AI SDK** | JS/TS 중심, Next.js 통합 | LangChain (Framework 레이어) |
| **CrewAI** | 역할 기반 멀티에이전트 | LangGraph + Supervisor 패턴 |
| **OpenAI Agents SDK** | OpenAI 전용 최적화 | LangChain + LangGraph |
| **Google ADK** | Google Cloud 통합 최적화 | LangChain + LangGraph |
| **LangChain** | 공급자 무관 통합 + 런타임 | 해당 없음 (세 계층 모두 포함) |


In [ ]:
from langchain.chat_models import init_chat_model
from langchain.messages import HumanMessage

    current_model = init_chat_model(provider_model)

# TODO: 이 셀의 핵심 구현 코드를 수업 시간에 직접 작성하세요.
# 목표: 공급자 독립성 데모: 같은 코드로 다른 모델 사용
# 위 마크다운 설명과 힌트를 참고해 필요한 타입, 함수, 그래프 구성, 실행 코드를 완성하세요.


## 7. 실습: 나의 프로젝트에 적합한 계층 선택하기


In [ ]:
# ============================================================
# TODO: 아래 시나리오들을 읽고 각각 어느 계층이 적합한지
#       변수에 값을 채워보세요
#
# 시나리오:
#   A. 빠르게 FAQ 챗봇 프로토타입을 만들어야 해요
#   B. 코드 리뷰 에이전트를 만들 건데, 파일을 읽고 수정해야 해요
#   C. 주문 처리 에이전트인데, 중요한 결제 전에 사람 승인이 필요해요
#   D. OpenAI에서 Anthropic으로 마이그레이션하기 쉬어야 해요
#
# 선택지: "LangChain", "LangGraph", "Deep Agents"
#
# 힌트:
#   - 파일시스템, 서브에이전트가 필요하면 → Deep Agents
#   - HITL, 상태 복구, 복잡한 분기가 필요하면 → LangGraph
#   - 빠른 시작, 단순 에이전트 → LangChain Framework
# ============================================================

# TODO: 각 변수에 적절한 계층을 문자열로 넣어보세요


## 핵심 요약

이 노트북에서 다음 내용을 배웠어요:

- **LangChain Framework**: 통합 + 추상화 레이어예요. `init_chat_model`, `create_agent`, `@tool`로 빠르게 에이전트를 만들 수 있어요. 공급자 독립성이 핵심 강점이에요.
- **LangGraph Runtime**: 저수준 오케스트레이션 엔진이에요. `StateGraph`, 체크포인터, `interrupt`로 복잡한 워크플로우를 정밀하게 제어할 수 있어요. LangChain Framework가 내부적으로 사용해요.
- **Deep Agents Harness**: 배터리 포함 에이전트 환경이에요. 파일시스템, 서브에이전트, 코드 실행 등 6가지 하네스 기능을 제공해요. 코딩 에이전트처럼 실제 시스템과 상호작용하는 에이전트에 적합해요.
- **계층 관계**: 세 계층은 배타적이지 않고 보완적이에요. Deep Agents는 LangChain과 LangGraph 위에 구축되어 있어요.
- **선택 기준**: 파일시스템/서브에이전트 필요 → Deep Agents, 복잡한 HITL/상태 관리 필요 → LangGraph, 빠른 시작 → LangChain Framework


## 참고: 에이전트 = 모델 + 모델 외부의 모든 것

LangChain의 3계층 위로 갈수록 두꺼워지는 "모델 외부 인프라"를 한 단어로 *harness* 라고 부르기도 합니다 (Martin Fowler, 2026). 이 교재는 그 인프라를 LangGraph·미들웨어·Skills·MCP로 직접 조립합니다. 용어 자체는 부차적이고, 본질은 "어떤 컴포넌트로 무엇을 보강할지"의 선택입니다.


## 다음 노트북 예고

다음 챕터 `02_LangGraph_Basics/01-LangGraph-Python-Basics.ipynb`에서는 **TypedDict, Annotated, add_messages** 를 배워요. LangGraph가 상태(State)를 어떻게 정의하고 관리하는지, Python 타입 시스템을 어떻게 활용하는지 기초부터 탄탄하게 다져봐요.
